In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [24]:
system_prompt = """

You are a personal chef. You will be given an image of leftover ingredients in the fridge.

First List the ingredients you see in the image.

Then, using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

model = init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct", model_provider="groq")
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
import base64
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [25]:
print(uploader.value)

({'name': 'fridge.jpg', 'type': 'image/jpeg', 'size': 48579, 'content': <memory at 0x114e9d6c0>, 'last_modified': datetime.datetime(2026, 4, 1, 8, 22, 2, 107000, tzinfo=datetime.timezone.utc)},)


In [26]:
if uploader.value:
    uploaded_file = uploader.value[0]
    img_bytes = uploaded_file["content"]
    mime_type = uploaded_file["type"]
else:
    # Fallback: load from a file path
    image_path = "resources/fridge.jpg"
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    mime_type = "image/jpeg"

img_b64 = base64.b64encode(img_bytes).decode("utf-8")
print(f"Image loaded ({mime_type}, {len(img_bytes)} bytes)")

Image loaded (image/jpeg, 48579 bytes)


In [27]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

message = HumanMessage(content=[
    {"type": "text", "text": "Here is a picture of my fridge. What can I make? Also, is there a tomatoe there ?"},
    {"type": "image", "base64": img_b64, "mime_type": mime_type},
])

response = agent.invoke(
    {"messages": [message]},
    config
)

print(response['messages'][-1].content)

The ingredients in your fridge are:

* Bananas
* Squash
* Yellow bell peppers
* Oranges
* Carrots
* Sweet potatoes
* Tomatoes
* Red bell peppers
* Watermelon
* Eggplant
* Onions
* Cabbage
* Blueberries
* Broccoli
* Cucumbers
* Limes
* Green grapes

As for your question, yes, there are tomatoes in the fridge.

Here are some recipe suggestions using these ingredients:

1. Summer Vegetable Recipes - The Spruce Eats
2. 20+ Tasty Recipes With Summer Fruits & Vegetables - Fannetastic Food
3. Colorful Veggie Delight with Dipping Sauce - Facebook

You can find more recipe ideas by searching online or checking out cookbooks. Let me know if you'd like me to provide recipe instructions for any of these suggestions!


In [28]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Here are some recipe suggestions using leftover chicken and rice:

1. Chicken and Rice Casserole - Crazy for Crust
2. Leftover Chicken and Egg Fried Rice - Easy Peasy Foodie
3. Chicken Fried Rice - Quick Weeknight Meals

You can find more recipe ideas by searching online or checking out cookbooks. Let me know if you'd like me to provide recipe instructions for any of these suggestions!


In [29]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Here is a picture of my fridge. What can I make? Also, is there a tomatoe there ?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}], additional_kwargs={}, response_metadata={}, id='f7bb27d7-7a17-4d2c-a47a-6122470dbdd7'),
              AIMessage(content='The ingredients I see in the image are:\n\n* Bananas\n* Squash\n* Yellow bell peppers\n* Oranges\n* Carrots\n* Sweet potatoes\n* Tomatoes\n* Red bell peppers\n* Watermelon\n* Eggplant\n* Onions\n* Cabbage\n* Blueberries\n* Broccoli\n* Cucumbers\n* Limes\n* Green grapes\n\nAs for your question about tomatoes, yes, I see several tomatoes in the fridge.\n\nNow, let me search the web for recipe suggestions using these ingredients.\n\n', additional_kwargs={'tool_calls': [{'id': 'y31sz83r1', 'function': {'arguments': '{"query":"recipes using bananas, squash, yellow bell peppers, oranges, carrots, sweet potatoes, tomatoes, red bell peppers, watermelon, eggplant, onio